In [20]:
from langchain_community.document_loaders import PyMuPDFLoader
import os
# 設定 PDF 檔案路徑
data_folder = "../data"

pdf_files = [f for f in os.listdir(data_folder) if f.endswith(".pdf")]
# 逐一處理每個 PDF，並存入獨立的變數
pdf_documents = {}  # 用來存放每個 PDF 的 documents
for pdf_file in pdf_files:
    pdf_path = os.path.join(data_folder, pdf_file)
    
    print(f"\n讀取 {pdf_file} ...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_documents[pdf_file] = loader.load()

    # 顯示該 PDF 總共頁數
    print(f"{pdf_file} 總共讀取 {len(pdf_documents[pdf_file])} 頁\n")


讀取 GI1.pdf ...
GI1.pdf 總共讀取 24 頁


讀取 L66.pdf ...
L66.pdf 總共讀取 26 頁


讀取 PX5.pdf ...
PX5.pdf 總共讀取 8 頁


讀取 DPA.pdf ...
DPA.pdf 總共讀取 10 頁


讀取 ZCN.pdf ...
ZCN.pdf 總共讀取 32 頁


讀取 CFE.pdf ...
CFE.pdf 總共讀取 25 頁


讀取 ZDN.pdf ...
ZDN.pdf 總共讀取 32 頁


讀取 FSAFSB.pdf ...
FSAFSB.pdf 總共讀取 9 頁


讀取 CFG.pdf ...
CFG.pdf 總共讀取 7 頁


讀取 SC1.pdf ...
SC1.pdf 總共讀取 9 頁


讀取 AGG.pdf ...
AGG.pdf 總共讀取 9 頁



In [3]:
import pdfplumber
import pandas as pd
from IPython.display import display
import os

data_folder = "../data"
output_root_folder = "../output"
os.makedirs(output_root_folder, exist_ok=True)  # 如果資料夾不存在則建立
# 找到所有 PDF 檔案
# pdf_files = [f for f in os.listdir(data_folder) if f.endswith(".pdf")]
# 找到特定 PDF 檔案
pdf_files = ["CFE.pdf"]


# 逐一處理每個 PDF
for pdf_file in pdf_files:
    pdf_path = os.path.join(data_folder, pdf_file)
    pdf_name = os.path.splitext(pdf_file)[0]  # 去掉副檔名
    output_folder = os.path.join(output_root_folder, pdf_name)  # 直接放在 output_root_folder
    os.makedirs(output_folder, exist_ok=True)  # 為每個 PDF 建立獨立的資料夾

    # 使用 pdfplumber 提取表格
    table_data = []
    table_titles = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            extracted_tables = page.extract_tables()

            # 如果該頁沒有表格，則跳過，不抓取「附表」
            if not extracted_tables:
                continue  

            # 解析頁面文本成行，確保標題來自表格的前一句
            text = page.extract_text()
            lines = text.split("\n") if text else []
            titles = [line.strip() for line in lines if "附表" in line]  # 找出所有「附表」或「表」


            # 確保表格數量與標題數量匹配
            while len(titles) < len(extracted_tables):
                titles.append("") # 沒有標題時補空字串，稍後改用 `Table_{page_num}_{索引+1}`
            
            # 儲存表格
            for table_idx, table in enumerate(extracted_tables):
                df = pd.DataFrame(table)
                table_data.append(df)

                # 依序分配標題
                title = titles[table_idx]
                if title:
                    table_title = f"Table{page_num}_{table_idx+1}_{title}"
                else:
                    table_title = f"Table{page_num}_{table_idx+1}"

                table_titles.append(table_title)

                print(f"\n處理 {pdf_file} - 第 {page_num} 頁 - 標題: {table_title}")

        
    # 儲存表格為 CSV
    if table_data:
        for i, df in enumerate(table_data):
            title = table_titles[i]
            safe_title = title.replace("：", "").replace(" ", "_").replace("/", "_")
            output_file = os.path.join(output_folder, f"{safe_title}.csv")
            df.to_csv(output_file, index=False, encoding="utf-8-sig")
            print(f"表格已儲存至 {output_file}")
    else:
        print(f"{pdf_file} 沒有偵測到表格")

    # # **使用 LangChain 讀取 PDF**
    # print(f"\n使用 LangChain 讀取 {pdf_file} ...")
    # loader = PyMuPDFLoader(pdf_path)
    # documents = loader.load()
    # print(f"總共讀取 {len(documents)} 頁\n")

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox



處理 國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型).pdf - 第 5 頁 - 標題: Table5_1_附表：短期費率表

處理 國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型).pdf - 第 5 頁 - 標題: Table5_2

處理 國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型).pdf - 第 6 頁 - 標題: Table6_1


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox



處理 國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型).pdf - 第 7 頁 - 標題: Table7_1
表格已儲存至 ../output/國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)/Table5_1_附表短期費率表.csv
表格已儲存至 ../output/國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)/Table5_2.csv
表格已儲存至 ../output/國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)/Table6_1.csv
表格已儲存至 ../output/國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)/Table7_1.csv


In [24]:
from langchain_community.document_loaders import PyMuPDFLoader
import os
import re

# 設定 PDF 資料夾
data_folder = "../data"

# 找到所有 PDF 檔案
# pdf_files = [f for f in os.listdir(data_folder) if f.endswith(".pdf")]
# 找到特定 PDF 檔案
pdf_files = ["CFG.pdf"]

# 例外表格
exception_keywords = ["(外溢型)年繳費率表", "國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)年繳費率表"]

# 用字典存放所有 PDF 解析結果
pdf_documents = {}

for pdf_file in pdf_files:
    pdf_path = os.path.join(data_folder, pdf_file)
    
    print(f"\n讀取 {pdf_file} ...")
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()  # 獨立解析此 PDF

    # 初始化解析結果
    pdf_lines = "\n".join([doc.page_content for doc in documents]).split("\n") 
    # 頁碼正則表達式，匹配 "第X頁，共Y頁"
    page_number_pattern = re.compile(r"^(第\s*\d+\s*頁[,，]?\s*共\s*\d+\s*頁)$")
    
    # 抓取 PDF 第一行作為 `insurance_name`
    insurance_name = ""
    for line in pdf_lines:
        line = line.strip()
        if line and not page_number_pattern.match(line):  # 跳過空行和頁碼行
            insurance_name = line
            break  # 取第一個有效行
    
    sections = []
    appendices = []
    metadata_dict = {
        "insurance_name": insurance_name,   
        "payout_items": "",     
        "contact_info": "",
        "summary": "",
        "other_metadata": [],   
        "announcement_history": []  
    }
    current_section = {"title": "", "content": ""}
    current_appendix = {"title": "", "content": ""}
    summary_started = False
    payout_items_started = False
    contact_info_started = False
    metadata_extracted = False
    appendix_started = False 
    appendix_title_pattern = re.compile(r"^(附表[\s\d一二三四五六七八九十]+|附件[\s\d一二三四五六七八九十]+|表|註\d+)[:：]")
    
    def is_appendix_title_line(line):
        return bool(appendix_title_pattern.match(line) or any(kw in line for kw in exception_keywords))
    
    # 解析 PDF 內容
    for line in pdf_lines:
        line = line.strip()  

        # 過濾掉頁碼
        if page_number_pattern.match(line):
            continue  
        
        # 檢測摘要範圍
        if "內容摘要" in line:
            summary_started = True
        if summary_started:
            if line == metadata_dict["insurance_name"]:  # 當遇到 `insurance_name`，停止摘要擷取
                summary_started = False
                metadata_dict["summary"] = metadata_dict["summary"].strip()  # 去除首尾空格
            else:  
                metadata_dict["summary"] += " " + line  # 直接存入摘要
            continue           
        
        # 進入附錄模式
        if appendix_started:
            if is_appendix_title_line(line):  
                if current_appendix["title"]:  
                    appendices.append(current_appendix)
                current_appendix = {"title": line, "content": ""}  
            else:
                current_appendix["content"] += line + " "  
            continue

        # 檢測附錄開始
        if re.match(r"^附[表件][一二三四五六七八九十0-9]*[:：]", line) or any(kw in line for kw in exception_keywords):
            appendix_started = True  
            if current_section["title"]:  
                sections.append(current_section)
            current_section = None  
            current_appendix = {"title": line, "content": ""}
            continue  

        # 檢測「給付項目」，並設定 `insurance_name`
        if "（給付項目：" in line or "（給付項目:" in line:
            payout_items_started = True
        if payout_items_started:
            metadata_dict["payout_items"] += " " + line
            if "）" in line:  # 結束條件
                payout_items_started = False
            continue  
        
        # 解析「聯絡資訊」
        if "（申訴電話：" in line:
            contact_info_started = True
        if contact_info_started:
            metadata_dict["contact_info"] += line
            if "tw）" in line:
                contact_info_started = False
            continue 

       # 解析「公告/修正紀錄」，只抓取包含「號函」的行
        if not metadata_extracted and "號函" in line:  
            metadata_dict["announcement_history"].append(line)
            continue 

        # 解析「其他 Metadata」
        if not metadata_extracted and "（" in line and "）" in line and not (payout_items_started or contact_info_started):
            metadata_dict["other_metadata"].append(line)

        # 偵測條款標題（Metadata 提取結束）
        if line.startswith("第") and "條" in line:
            metadata_extracted = True  
            if current_section["title"]:  
                sections.append(current_section)
            current_section = {"title": line, "content": ""}  
        elif metadata_extracted:
            current_section["content"] += line   

    # 加入最後的條款和附錄
    if current_section and current_section["title"]:
        sections.append(current_section)
    if current_appendix["title"]:
        appendices.append(current_appendix)

    # 存入字典
    pdf_documents[pdf_file] = {
        "metadata": metadata_dict,
        "sections": sections,
        "appendices": appendices
    }

    # 顯示解析結果
    print("\nMetadata 解析結果:")
    print(f"保險產品名稱: {metadata_dict['insurance_name']}")
    print(f"給付項目: {metadata_dict['payout_items']}")
    print(f"聯絡資訊: {metadata_dict['contact_info']}")
    print(f"公告/修正紀錄: {metadata_dict['announcement_history']}\n")
    
    print(f"解析出 {len(sections)} 條款")
    for i, sec in enumerate(sections[:3]):  
        print(f"\n{sec['title']}")
        print(sec["content"][:300])  

    if appendices:
        print("\n附錄內容：")
        for i, app in enumerate(appendices[:3]):  
            print(f"\n{app['title']}")
            print(app["content"][:300])

    print("\n" + "=" * 50)  # 分隔不同 PDF 的解析結果



讀取 CFG.pdf ...

Metadata 解析結果:
保險產品名稱: 國泰人壽自由配一年定期初次罹患重大疾病健康保險(甲型)
給付項目:  （給付項目：重大疾病保險金）
聯絡資訊: （申訴電話：市話免費撥打0800-036-599、付費撥打02-2162-6201；傳真：0800-211-568；電子信箱（E-mail）：service@cathaylife.com.tw）
公告/修正紀錄: ['110.08.31 國壽字第1100081182 號函備查', '112.02.08 依111.08.30 金管保壽字第1110445485 號函修正']

解析出 23 條款

第一條 保險契約的構成
本保險單條款、附著之要保書、批註及其他約定書，均為本保險契約（以下簡稱本契約）的構成部分。本契約的解釋，應探求契約當事人的真意，不得拘泥於所用的文字；如有疑義時，以作有利於被保險人的解釋為原則。

第二條 名詞定義
本契約名詞定義如下：ㄧ、「意外傷害事故」：指非由疾病引起之外來突發事故。二、「重大疾病」：係指被保險人經醫院醫師診斷確定而屬下列情形之一者為限：(一)腦中風後障礙(重度)係指因腦血管的突發病變導致腦血管出血、栓塞、梗塞致永久性神經機能障礙者。所謂永久性神經機能障礙係指事故發生六個月後經神經科、神經外科或復健科專科醫師認定仍遺留下列機能障礙之一者：1.植物人狀態。2.一上肢三大關節或一下肢三大關節遺留下列機能障礙之一者：(1)關節機能完全不能隨意識活動。(2)肌力在2分(含)以下者(肌力2分是指可做水平運動，但無法抗地心引力)。上肢三大關節包括肩、肘、腕關節，下肢三大關節包括髖、膝、踝關節。3.

第三條 保險責任的開始及交付保險費
本公司應自同意承保並收取保險費後負保險責任，並應發給保險單作為承保的憑證。本公司如於同意承保前，預收相當於保險費之金額時，其應負之保險責任，以同意承保時溯自預收相當於保險費金額時開始。前項情形，在本公司為同意承保與否之意思表示前發生應予給付之保險事故時，本公司仍負保險責任。

附錄內容：

附表：短期費率表
一、一個月以上短期費率 月數 1 2 3 4 5 6 7 8 9 10 11 12 對年繳保費比(%) 15 25 35 45 55 65 75 80 85 90 95 100  二、未滿一個月短期費率 日數 1 2 3

In [25]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

# 設定 chunking 規則
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=15
)

# 為每個 PDF 進行 chunking
for pdf_file, data in pdf_documents.items():
    sections = data["sections"]
    appendices = data["appendices"]

    chunked_sections = []
    chunked_appendices = []

    section_chunk_id = 0

    # 建立 title -> group 對照表（條款）
    title_to_group = {}
    group_counter = 0
    for sec in sections:
        title = sec["title"]
        if title not in title_to_group:
            title_to_group[title] = f"section-{group_counter:03d}"
            group_counter += 1

    # 處理條款
    for sec in sections:
        title = sec["title"]
        group_id = title_to_group[title]
        chunks = text_splitter.split_text(sec["content"])

        for chunk in chunks:
            related_appendix = None  # 先空著，稍後會處理

            chunked_sections.append({
                "title": title,
                "content": chunk,
                "chunk_id": section_chunk_id,
                "group": group_id,
                "related_appendix": related_appendix
            })

            section_chunk_id += 1

    # 接續 chunk_id 編號
    appendix_chunk_id = section_chunk_id

    # 建立 title -> group 對照表（附錄）
    appendix_title_to_group = {}
    appendix_group_counter = 0
    for app in appendices:
        title = app["title"]
        if title not in appendix_title_to_group:
            appendix_title_to_group[title] = f"appendix-{appendix_group_counter:03d}"
            appendix_group_counter += 1

    # 建立 appendix 表號 → chunk_id 對照表（只記錄第一個 chunk）
    appendix_key_to_chunk_id = {}

    # 處理附錄
    for app in appendices:
        title = app["title"]
        group_id = appendix_title_to_group[title]
        chunks = text_splitter.split_text(app["content"])

        for idx, chunk in enumerate(chunks):
            if idx == 0:
                match = re.search(r"(附表[:：]?\s*[\u4e00-\u9fa5]*表[一二三四五六七八九十百千萬]*|[\u4e00-\u9fa5]*表[一二三四五六七八九十百千萬]*)", title)
                if match:
                    appendix_key = match.group(1)
                    appendix_key_to_chunk_id[appendix_key] = appendix_chunk_id

            chunked_appendices.append({
                "title": title,
                "content": chunk,
                "chunk_id": appendix_chunk_id,
                "group": group_id,
            })

            appendix_chunk_id += 1

    # 再回頭補上 section 的 related_appendix（這段比對放最後比較穩）
    for section in chunked_sections:
        for appendix_key, appendix_chunk in appendix_key_to_chunk_id.items():
            if appendix_key in section["content"]:
                section["related_appendix"] = appendix_chunk
                break

    # 存入 `pdf_documents`
    pdf_documents[pdf_file]["sections"] = chunked_sections
    pdf_documents[pdf_file]["appendices"] = chunked_appendices

    # 顯示 chunking 結果
    print(f"\n{pdf_file} 總共切分成 {len(chunked_sections)} 個條款 chunk")
    for sec in chunked_sections[:3]:
        print(f"\nchunk_id: {sec['chunk_id']} | {sec['title']}")
        print(f"group: {sec['group']}")
        print(f"related_appendix: {sec['related_appendix']}")
        print(sec["content"][:200])

    print(f"\n{pdf_file} 總共切分成 {len(chunked_appendices)} 個附錄 chunk")
    for app in chunked_appendices[:3]:
        print(f"\nchunk_id: {app['chunk_id']} | {app['title']}")
        print(f"group: {app['group']}")
        print(app["content"][:200])

    print("\n" + "=" * 50)



CFG.pdf 總共切分成 24 個條款 chunk

chunk_id: 0 | 第一條 保險契約的構成
group: section-000
related_appendix: None
本保險單條款、附著之要保書、批註及其他約定書，均為本保險契約（以下簡稱本契約）的構成部分。本契約的解釋，應探求契約當事人的真意，不得拘泥於所用的文字；如有疑義時，以作有利於被保險人的解釋為原則。

chunk_id: 1 | 第二條 名詞定義
group: section-001
related_appendix: None
本契約名詞定義如下：ㄧ、「意外傷害事故」：指非由疾病引起之外來突發事故。二、「重大疾病」：係指被保險人經醫院醫師診斷確定而屬下列情形之一者為限：(一)腦中風後障礙(重度)係指因腦血管的突發病變導致腦血管出血、栓塞、梗塞致永久性神經機能障礙者。所謂永久性神經機能障礙係指事故發生六個月後經神經科、神經外科或復健科專科醫師認定仍遺留下列機能障礙之一者：1.植物人狀態。2.一上肢三大關節或一下肢三大關節遺

chunk_id: 2 | 第二條 名詞定義
group: section-001
related_appendix: None
而導致部分心肌壞死，其診斷除了發病90日(含)後，經心臟影像檢查證實左心室功能射出分率低於50%(含)者之外，且同時具備下列至少二個條件：1.典型之胸痛症狀。2.最近心電圖的異常變化，顯示有心肌梗塞者。3.心肌酶CK-MB有異常增高，或肌鈣蛋白T>1.0ng/ml，或肌鈣蛋白I>0.5ng/ml。(五)冠狀動脈繞道手術係指因冠狀動脈疾病而有持續性心肌缺氧造成心絞痛或心臟衰竭，並接受冠狀動脈繞道手術

CFG.pdf 總共切分成 4 個附錄 chunk

chunk_id: 24 | 附表：短期費率表
group: appendix-000
一、一個月以上短期費率 月數 1 2 3 4 5 6 7 8 9 10 11 12 對年繳保費比(%) 15 25 35 45 55 65 75 80 85 90 95 100  二、未滿一個月短期費率 日數 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 對年繳保費比(%) 5.0 5.3 5.7 6.0 6.4 6.7 7.1 7.4 7.8 8.1 8.4 8.8 

In [26]:
import json
import os

# 設定輸出資料夾
output_root_folder = "../output"

# 逐一處理每個 PDF，將 chunking 結果存入對應的輸出資料夾
for pdf_file, data in pdf_documents.items():
    pdf_name = os.path.splitext(pdf_file)[0]  # 去掉 `.pdf` 副檔名
    output_folder = os.path.join(output_root_folder, pdf_name)  # 每個 PDF 獨立的輸出資料夾
    os.makedirs(output_folder, exist_ok=True)  # 確保資料夾存在

    output_file = os.path.join(output_folder, "pdf_parsing.json")  # JSON 儲存路徑

    # 準備 JSON 輸出資料
    output_data = {
        "metadata": data["metadata"],  
        "sections": data["sections"],  
        "appendices": data["appendices"]
    }

    # 儲存 JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=4)

    print(f"已將 {pdf_file} 的切分結果存入 {output_file}")


已將 CFG.pdf 的切分結果存入 ../output/CFG/pdf_parsing.json
